# Scraper de calidad del aire

Ejecuta solo el scraping que usa la app y muestra la tabla resultante. No guarda archivos, no calcula predicciones y no sube nada a GitHub. El codigo esta completo en el notebook: no importa funciones de `app/pipeline.py`.

In [ ]:
from io import StringIO

import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait


SCRAPER_URL = "https://www.valencia.es/valenciaalminut/"
SCRAPER_TABLE_ID = "tabla_dinamica"
SCRAPER_COLUMNS = ["µg/m3", "SO2", "NO2", "O3", "PM-10", "PM-2.5"]


def scrape_current_table() -> pd.DataFrame:
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-extensions")
    options.add_argument("--disable-popup-blocking")
    options.add_argument("--disable-notifications")
    options.add_argument("--ignore-certificate-errors")

    driver = webdriver.Chrome(options=options)
    try:
        driver.get(SCRAPER_URL)
        table_html = WebDriverWait(driver, 40).until(
            EC.presence_of_element_located((By.ID, SCRAPER_TABLE_ID))
        ).get_attribute("outerHTML")
    finally:
        driver.quit()

    df = pd.read_html(StringIO(table_html))[0]
    df = df[SCRAPER_COLUMNS].dropna(how="all")
    if df.empty or df["µg/m3"].dropna().empty:
        raise RuntimeError("Scrape returned no station rows.")
    return df

df = scrape_current_table()
print(df.to_string(index=False))